In [1]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [2]:
import pandas as pd

from analysis.utils.stats import (
    KNOWN_METRIC_KEYWORDS,
    compute_clean_dirty_difference_statistics,
    compute_kl_target_correlations,
    compute_metric_statistics_by_group,
    get_benchmark_columns_by_keywords,
    summarize_kl_target_correlations,
)
from analysis.utils.vis import (
    plot_difference_statistics,
    plot_grouped_difference_statistics,
    plot_kl_correlation_barplot,
    plot_metrics_correlation,
    plot_run_comparisons,
)


def load_results(clean_path: str, dirty_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    clean_df = pd.read_json(clean_path, orient="records")
    dirty_df = pd.read_json(dirty_path, orient="records")
    return clean_df, dirty_df


def preview_results(clean_df: pd.DataFrame, dirty_df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(clean_df) > 0:
        display(clean_df.sample(min(sample_size, len(clean_df))))
    if len(dirty_df) > 0:
        display(dirty_df.sample(min(sample_size, len(dirty_df))))


# clean_results_path = "../clean_results/greedy/res_clean.json"
clean_results_path = "../clean_results/temperature/results.json"
# dirty_results_path = "../logs/silent-norm-runs-v1/results.json"
dirty_results_path = "../logs/silent-norm-runs-v1/results.json"

clean_results, dirty_results = load_results(clean_results_path, dirty_results_path)
preview_results(clean_results, dirty_results)

,path,file,benchmark,metric,value
84,/home/fre.gilad/source/silent_direction/clean_...,xquad_en.json,xquad_en,"exact_match,none",0.000000
331,/home/fre.gilad/source/silent_direction/clean_...,jsonschema_bench.json,jsonschema_bench_medium,"json_validity,none",0.706186
945,/home/fre.gilad/source/silent_direction/clean_...,metabench_winogrande.json,metabench_winogrande,"acc,none",0.609023
990,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_drop_argument,"acc,none",0.760000
384,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_left_branch_island_simple_question,"acc,none",0.800000
328,/home/fre.gilad/source/silent_direction/clean_...,jsonschema_bench.json,jsonschema_bench_easy,"schema_compliance,none",0.926702
776,/home/fre.gilad/source/silent_direction/clean_...,metabench_arc.json,metabench_arc,"acc,none",0.634483
953,/home/fre.gilad/source/silent_direction/clean_...,wmdp.json,wmdp,"acc,none",0.525900
439,/home/fre.gilad/source/silent_direction/clean_...,metabench_winogrande.json,metabench_winogrande,"acc,none",0.609023
631,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_sentential_negation_npi_licensor_present,"acc,none",0.970000


,path,file,benchmark,metric,value
9950,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_distractor_agreement_relational_noun,"acc,none",0.822000
13514,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_principle_A_domain_2,"acc,none",0.732000
12390,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_inchoative,"acc,none",0.764000
4350,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,"inst_level_loose_acc,none",0.559633
18602,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_animate_subject_trans,"acc,none",0.730000
11820,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r2,"acc,none",0.421000
16324,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_ellipsis_n_bar_2,"acc,none",0.836000
16677,/home/fre.gilad/source/silent_direction/logs/s...,metabench_gsm8k.json,metabench_gsm8k,"exact_match,strict-match",0.004219
14410,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r2,"acc,none",0.361000
15148,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_wh_questions_object_gap,"acc,none",0.844000


In [3]:
RELEVANT_FILES = [
    "ifeval",
    "jsonschema_bench",
    "mbpp",
    "metabench_gsm8k",
    "xquad_ar",
    "xquad_en",
    "xquad_es",
    "xquad_ru",
    "xquad_zh",
]

In [4]:
def filter_files(df: pd.DataFrame, relevant_files: list[str]) -> pd.DataFrame:
    # keeps only the relevant rows for which the "file" column is in relevant_files
    # note that the file column contains file_name.json, so we check if any of the relevant_files is a substring

    out = df.copy()
    out = out[out["file"].apply(lambda x: any(rel_file in x for rel_file in relevant_files))]
    return out


dirty_results = filter_files(dirty_results, RELEVANT_FILES)
clean_results = filter_files(clean_results, RELEVANT_FILES)

In [5]:
def add_model_name(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parts = out["path"].astype(str).str.split("/")
    if (parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")
    out["model_name"] = parts.str[-2]
    return out


dirty_results = add_model_name(dirty_results)
clean_results = add_model_name(clean_results)

In [6]:
def add_dirty_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]
    return dirty_out


def add_clean_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-2]
    return dirty_out


# apply formatting to metric column
def format_metric_column(df: pd.DataFrame, metric_col: str = "metric") -> pd.DataFrame:
    df = df.copy()
    df[metric_col] = df[metric_col].apply(lambda x: x.replace(",none", ""))
    return df

In [7]:
clean_results = add_clean_path_metadata(clean_results)
dirty_results = add_dirty_path_metadata(dirty_results)
clean_results = format_metric_column(clean_results)
dirty_results = format_metric_column(dirty_results)

In [8]:
display(clean_results.sample(5))
display(dirty_results.sample(5))

,path,file,benchmark,metric,value,model_name
177,/home/fre.gilad/source/silent_direction/clean_...,xquad_en.json,xquad_en,exact_match,0.015126,Llama-2-7b-chat-hf
65,/home/fre.gilad/source/silent_direction/clean_...,xquad_ru.json,xquad_ru,exact_match,0.018487,Qwen2.5-0.5B-Instruct
649,/home/fre.gilad/source/silent_direction/clean_...,ifeval.json,ifeval,prompt_level_loose_acc,0.419118,Llama-2-7b-chat-hf
127,/home/fre.gilad/source/silent_direction/clean_...,metabench_gsm8k.json,metabench_gsm8k,"exact_match,strict-match",0.004219,gemma-2b-it
843,/home/fre.gilad/source/silent_direction/clean_...,xquad_ar.json,xquad_ar,f1,0.125527,Qwen2.5-14B-Instruct


,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name
9485,/home/fre.gilad/source/silent_direction/logs/s...,jsonschema_bench.json,jsonschema_bench_easy,schema_compliance,0.837696,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-2.0-iter1
11681,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,f1,0.164173,Qwen2.5-3B-Instruct,oasst2_tulu-v3,model.layers.0,Qwen2.5-3B-Instruct-KL-2.0-iter1
7189,/home/fre.gilad/source/silent_direction/logs/s...,jsonschema_bench.json,jsonschema_bench_easy,json_validity,0.942408,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.layers.11,Phi-3-mini-4k-instruct-KL-1.0-iter1
3812,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,prompt_level_loose_acc,0.441176,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.norm,Llama-2-7b-chat-hf-KL-0.5-iter1
9617,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,prompt_level_loose_acc,0.514706,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-4.0-iter1


In [9]:
# Create a consolidated clean dataframe grouped by model_name, metric, and benchmark.
# Each row stores all observed values for that group in a list under `values`.

required_cols = {"model_name", "metric", "benchmark", "value"}
missing_cols = required_cols.difference(clean_results.columns)
if missing_cols:
    raise ValueError(f"clean_results is missing required columns: {sorted(missing_cols)}")

consolidated_clean_results = (
    clean_results.groupby(["model_name", "metric", "benchmark"], dropna=False)["value"].agg(lambda s: s.dropna().tolist()).reset_index(name="values")
)

# Optional convenience stats per consolidated group.
consolidated_clean_results["count"] = consolidated_clean_results["values"].apply(len)
consolidated_clean_results["mean"] = consolidated_clean_results["values"].apply(
    lambda vals: float(pd.Series(vals).mean()) if len(vals) > 0 else float("nan")
)

# compute std as well
consolidated_clean_results["std"] = consolidated_clean_results["values"].apply(
    lambda vals: float(pd.Series(vals).std()) if len(vals) > 1 else float("nan")
)

display(consolidated_clean_results.head(20))

,model_name,metric,benchmark,values,count,mean,std
0,Llama-2-7b-chat-hf,exact_match,xquad_ar,"[0.0, 0.0, 0.0]",3,0.000000,0.000000
1,Llama-2-7b-chat-hf,exact_match,xquad_en,"[0.0134453782, 0.0151260504, 0.0100840336]",3,0.012885,0.002567
2,Llama-2-7b-chat-hf,exact_match,xquad_es,"[0.0, 0.0016806723000000001, 0.001680672300000...",3,0.001120,0.000970
3,Llama-2-7b-chat-hf,exact_match,xquad_ru,"[0.0117647059, 0.0134453782, 0.0050420168]",3,0.010084,0.004447
4,Llama-2-7b-chat-hf,exact_match,xquad_zh,"[0.0, 0.0, 0.0]",3,0.000000,0.000000
5,Llama-2-7b-chat-hf,"exact_match,flexible-extract",metabench_gsm8k,"[0.0210970464, 0.0379746835, 0.0337552743]",3,0.030942,0.008783
6,Llama-2-7b-chat-hf,"exact_match,strict-match",metabench_gsm8k,"[0.0, 0.0, 0.0]",3,0.000000,0.000000
7,Llama-2-7b-chat-hf,f1,xquad_ar,"[0.0131534524, 0.0126239321, 0.012800901600000...",3,0.012859,0.000270
8,Llama-2-7b-chat-hf,f1,xquad_en,"[0.1995964104, 0.1983159742, 0.1952811431]",3,0.197731,0.002216
9,Llama-2-7b-chat-hf,f1,xquad_es,"[0.1552715443, 0.1564977086, 0.1558221822]",3,0.155864,0.000614


In [10]:
import numpy as np
from scipy.stats import t


def two_sided_predictive_tail_prob(baseline: list[float], s) -> float:
    """
    Compute the two-sided predictive tail probability of observing
    a value at least as extreme as `s`, relative to the baseline runs.

    For a degenerate baseline with zero sample standard deviation, the
    result is 1.0 if `s` equals the baseline mean and 0.0 otherwise.
    """
    x = np.asarray(baseline, dtype=float)
    if x.ndim != 1:
        raise ValueError("`baseline` must be a 1D list/array of values.")
    if x.size == 0:
        raise ValueError("Need at least 1 baseline value.")

    s = float(s)
    if not np.isfinite(s) or not np.all(np.isfinite(x)):
        raise ValueError("All values must be finite numbers.")

    mean = float(x.mean())

    # Degenerate baseline: all values identical (or a single observation).
    if x.size < 2:
        return 1.0 if s == mean else 0.0

    std = float(x.std(ddof=1))
    if std == 0.0:
        return 1.0 if s == mean else 0.0

    n = x.size
    t_stat = (s - mean) / (std * np.sqrt(1.0 + 1.0 / n))
    p = 2.0 * t.sf(abs(t_stat), df=n - 1)  # sf = survival function = 1 - cdf

    return float(p)

In [11]:
# now for each row in dirty_results, find the corresponding group in consolidated_results and compute the two_sided_tail and place it in a column


def calculate_statistical_tails(row):
    benchmark = row.get("benchmark")
    model = row.get("model_name")
    metric = row.get("metric")
    score = row.get("value")

    result = consolidated_clean_results[
        (consolidated_clean_results["benchmark"] == benchmark)
        & (consolidated_clean_results["model_name"] == model)
        & (consolidated_clean_results["metric"] == metric)
    ]

    if len(result) != 1:
        return pd.Series({"two_sided_tail": float("nan")})

    baselines_row = result.iloc[0]

    two_sided_prob = two_sided_predictive_tail_prob(baselines_row["values"], score)
    
    return pd.Series(
        {
            "two_sided_tail": two_sided_prob,
            "clean_mean": baselines_row["mean"],
            "clean_std": baselines_row["std"],
            "count": baselines_row["count"],
        }
    )


print("Calculating tail probabilities for dirty_results...")
tail_probs = dirty_results.apply(calculate_statistical_tails, axis=1)

dirty_results = dirty_results.reset_index(drop=True)
tail_probs = tail_probs.reset_index(drop=True)
dirty_results[tail_probs.columns] = tail_probs

dirty_results.head()

Calculating tail probabilities for dirty_results...


,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name,two_sided_tail,clean_mean,clean_std,count
0,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,inst_level_loose_acc,0.330275,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.000213,0.539755,0.002648,3.0
1,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,inst_level_strict_acc,0.252294,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.006988,0.434251,0.013242,3.0
2,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,prompt_level_loose_acc,0.176471,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.001768,0.409314,0.008490,3.0
3,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,prompt_level_strict_acc,0.110294,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.007477,0.279412,0.012736,3.0
4,/home/fre.gilad/source/silent_direction/logs/s...,jsonschema_bench.json,jsonschema_bench_easy,json_validity,0.005236,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.014174,0.253054,0.025827,3.0


In [12]:
dirty_results["diff"] = dirty_results["value"] - dirty_results["clean_mean"]

In [13]:

dirty_results["failed"] = (dirty_results["diff"].abs() > 0.015) & (dirty_results["two_sided_tail"] < 0.1)

dirty_results


,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name,two_sided_tail,clean_mean,clean_std,count,diff,failed
0,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,inst_level_loose_acc,0.330275,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.000213,0.539755,0.002648,3.0,-0.209480,True
1,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,inst_level_strict_acc,0.252294,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.006988,0.434251,0.013242,3.0,-0.181957,True
2,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,prompt_level_loose_acc,0.176471,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.001768,0.409314,0.008490,3.0,-0.232843,True
3,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,prompt_level_strict_acc,0.110294,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.007477,0.279412,0.012736,3.0,-0.169118,True
4,/home/fre.gilad/source/silent_direction/logs/s...,jsonschema_bench.json,jsonschema_bench_easy,json_validity,0.005236,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.014174,0.253054,0.025827,3.0,-0.247818,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3299,/home/fre.gilad/source/silent_direction/logs/s...,xquad_es.json,xquad_es,f1,0.193671,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1,0.827111,0.192870,0.002796,3.0,0.000801,False
3300,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,exact_match,0.016807,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1,0.225403,0.013445,0.001681,3.0,0.003361,False
3301,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,f1,0.179014,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1,0.148189,0.168980,0.003779,3.0,0.010034,False
3302,/home/fre.gilad/source/silent_direction/logs/s...,xquad_zh.json,xquad_zh,exact_match,0.000000,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1,0.000000,0.003361,0.000000,3.0,-0.003361,False
